# Introduction: Shape Classification with Equivariant Networks

This notebook introduces the project and walks through the full pipeline:
dataset generation → training → evaluation → result analysis.

We benchmark **12 neural network architectures** on **6 point cloud classification tasks**,
comparing permutation-invariant baselines (PointNet, DeepSets) against
equivariant Tensor Field Network variants.

## 1. Project Overview

| Category | Models | Key Idea |
|----------|--------|----------|
| **Baselines** | PersNet, RipsPointNet, ScalarInputMLP, ScalarDistanceDeepSet | Permutation-invariant, no geometric structure |
| **TFN** | TensorFieldNetwork, GTTFNv2, HierarchicalTFN, StochasticTFN | SO(3)-equivariant, respects rotation symmetry |
| **O(n)-equivariant** | OnEquivTFN, AttentionTFN, RelaxedTFN, HybridTFN | O(n) parity-equivariant, handles reflections |

Datasets range from simple 2D circles (3 classes) to complex 3D shapes (8 classes),
each with a clean and noisy (50% outlier corruption) variant.

## 2. Setup and Dataset Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets.utils import create_multiple_circles
from datasets.shapes3d import generate_dataset, DATASET_CONFIGS

In [ ]:
# 2D circles — clean and noisy
data_clean, labels_clean = create_multiple_circles(3, 600, noisy=False, N_noise=200)
data_noisy, labels_noisy = create_multiple_circles(3, 600, noisy=True, N_noise=200)

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for i, (d, l, tag) in enumerate([(data_clean, labels_clean, 'clean'),
                                   (data_noisy, labels_noisy, 'noisy')] * 3):
    row = i // 3
    idx = i % 3
    pc = d[idx]
    axes[i].scatter(pc[:, 0], pc[:, 1], s=1, alpha=0.5, c=plt.cm.tab10(idx))
    axes[i].set_title(f'{tag}: {l[idx]}', fontsize=10)
    axes[i].set_aspect('equal')
plt.suptitle('2D Circle Datasets', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 3D shapes — one example per dataset
fig, axes = plt.subplots(1, 4, figsize=(16, 4), subplot_kw={'projection': '3d'})
for ax, (name, cfg) in zip(axes, list(DATASET_CONFIGS.items())[:4]):
    data, labels, _, _, classes = generate_dataset(name, 5, 10, 600, noise_sigma=0.0, seed=42)
    pc = data[0]
    ax.scatter(pc[:, 0], pc[:, 1], pc[:, 2], s=1, alpha=0.4)
    ax.set_title(f'{name}\n({len(classes)} classes)', fontsize=10)
    ax.set_box_aspect([1, 1, 1])
plt.suptitle('3D Shape Datasets (sample point clouds)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 3. Training a Single Experiment

The main training script `shape/train_shape.py` handles all models and datasets.
Arguments: `<dataset> <model> <num_epochs> <trial> [batch_size]`

In [ ]:
# Example: train TFN on circles for 10 epochs, trial 0
!python shape/train_shape.py circles TensorFieldNetwork 10 0

In [ ]:
import json, glob, os

# Find the result we just generated
files = sorted(glob.glob('shape/results/shape_circles_TensorFieldNetwork_*.json'))
if files:
    with open(files[-1]) as f:
        result = json.load(f)
    print(f'Dataset:     {result["dataset"]}')
    print(f'Model:       {result["model"]}')
    print(f'Trial:       {result["trial"]}')
    print(f'Best val:    {result["best_val_accuracy"]:.1%}')
    print(f'Clean test:  {result["clean_accuracy"]:.1%}')
    print(f'Noisy test:  {result["noisy_accuracy"]:.1%}')
else:
    print('No result files found — check that the training completed.')

## 4. Batch Results

All experiments produce JSON files in `shape/results/`. Run all experiments on the cluster
with `shape/check_missing.py` and consolidate results with `shape/consolidate_results.py`.

For the full analysis, see **`shape_classification.ipynb`**.

In [ ]:
import pandas as pd

files = sorted(glob.glob('shape/results/shape_*.json'))
print(f'Found {len(files)} result files')

records = []
for f in files:
    with open(f) as fh:
        records.append(json.load(fh))

df = pd.DataFrame(records)
if len(df) > 0:
    summary = df.groupby(['dataset', 'model']).agg(
        val_mean=('best_val_accuracy', 'mean'),
        val_std=('best_val_accuracy', 'std'),
        clean_mean=('clean_accuracy', 'mean'),
        clean_std=('clean_accuracy', 'std'),
        noisy_mean=('noisy_accuracy', 'mean'),
        noisy_std=('noisy_accuracy', 'std'),
    ).reset_index()
    pivot = summary.pivot_table(index='model', columns='dataset',
                                 values='clean_mean', aggfunc='first').round(3)
    display(pivot)
else:
    print('No results to display yet.')

## 5. Reproducing

```bash
# Run a single experiment locally
python shape/train_shape.py circles TensorFieldNetwork 50 0

# Submit all experiments to cluster
cd shape && python check_missing.py
sbatch submit_resubmit.sh

# Consolidate results into CSV
python shape/consolidate_results.py
```